In [1]:
import os 
import time 
import numpy as np
import pandas as pd
from iminuit import Minuit
from scipy.special import j0
import plotly.graph_objects as go
from iminuit.cost import LeastSquares
from scipy.integrate import fixed_quad

In [2]:
# experimental data
save_folder = 'run7'
n_points = 10000

lower_factor = 0.99
upper_factor = 2 - lower_factor

# Load experimental data
atlas_data = pd.read_csv('../../../../../data/ens_atlas_difc0_2.dat', delim_whitespace=True, header=None)
totem_data = pd.read_csv('../../../../../data/ens_totem_difc0_2.dat', delim_whitespace=True, header=None)

# Function to process data for each experiment
def process_data(data, energy_blocks):
    x_values = []
    y_values = []
    y_errors = []
    
    for start, end in energy_blocks:
        block = data.iloc[start:end] if end is not None else data.iloc[start:]
        x_values.append(block[0].values)
        y_values.append(block[1].values)
        y_errors.append(block[2].values)
    
    return x_values, y_values, y_errors

# Energy ranges for each experiment (7TeV, 8TeV, 13TeV)
atlas_blocks = [(0, 29), (29, 58), (58, None)]
totem_blocks = [(0, 65), (65, 118), (118, None)]

# Process data
x_atlas, y_atlas, yerr_atlas = process_data(atlas_data, atlas_blocks)

# Extract values by energy (index 0=7TeV, 1=8TeV, 2=13TeV)
x_7_atlas, y_7_atlas, yerr_7_atlas = x_atlas[0], y_atlas[0], yerr_atlas[0]
x_8_atlas, y_8_atlas, yerr_8_atlas = x_atlas[1], y_atlas[1], yerr_atlas[1]
x_13_atlas, y_13_atlas, yerr_13_atlas = x_atlas[2], y_atlas[2], yerr_atlas[2]

/tmp/ipykernel_10294/1982994368.py:9: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  atlas_data = pd.read_csv('../../../../../data/ens_atlas_difc0_2.dat', delim_whitespace=True, header=None)
/tmp/ipykernel_10294/1982994368.py:10: FutureWarning: The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead
  totem_data = pd.read_csv('../../../../../data/ens_totem_difc0_2.dat', delim_whitespace=True, header=None)


In [3]:
# setting parameters
b_0 = (33 - 6) / (12 * np.pi)
Lambda = 0.284  # ΛQCD in GeV
gamma_1 = 0.084
gamma_2 = 2.36
rho = 4.0
s0 = 1.0
alpha_prime = 0.25



ensemble_parameters = {
    'atlas': {
        'log': {
            'epsilon': 0.0753,
            'mg': 0.356,
            'a1': 1.373,
            'a2': 2.50
        },
        'pl': {
            'epsilon': 0.0753,
            'mg': 0.421,
            'a1': 1.517,
            'a2': 2.05
        }
    },
    'totem': {
        'log': {
            'epsilon': 0.0892,
            'mg': 0.380,
            'a1': 1.491,
            'a2': 2.77
        },
        'pl':{
            'epsilon': 0.0892,
            'mg': 0.447,
            'a1': 1.689,
            'a2': 1.7
        }
    }
}

ensemble_atlas = 'atlas'  
ensemble_totem = 'totem'

log_model_type = 'log'
pl_model_type = 'pl'   

def get_parameters_with_variations(ensemble_parameters, ensemble_name, model_type, lower_factor=lower_factor, upper_factor=upper_factor):
    # Obtém os parâmetros iniciais
    initial_params = ensemble_parameters[ensemble_name][model_type]
    
    # Cria as variações
    initial_params_low = {k: v * lower_factor for k, v in initial_params.items()}
    initial_params_high = {k: v * upper_factor for k, v in initial_params.items()}
    
    return initial_params, initial_params_low, initial_params_high

# Get parameters for selected configuration
initial_params_pl_atlas = ensemble_parameters[ensemble_atlas][pl_model_type]

# Para Atlas
initial_params_pl_atlas, initial_params_low_pl_atlas, initial_params_high_pl_atlas = \
    get_parameters_with_variations(ensemble_parameters, ensemble_atlas, pl_model_type)




In [4]:
# def model functions 
def m2_log(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = np.log((q2 + rho_mg_squared) / lambda_squared) / np.log(rho_mg_squared / lambda_squared)
    return mg ** 2 * ratio ** (-1 - gamma_1)

def m2_pl(q2, mg):
    lambda_squared = Lambda ** 2
    rho_mg_squared = rho * mg ** 2
    ratio = np.log((q2 + rho_mg_squared) / lambda_squared) / np.log(rho_mg_squared / lambda_squared)
    return (mg ** 4 / (q2 + mg ** 2)) * ratio ** (gamma_2 - 1)


def G_p(q2, a1, a2):
    return np.exp(-(a1 * q2 + a2 * q2 ** 2))

def alpha_D(q2, mg, m2_func):
    m2 = m2_func(q2, mg)
    return 1.0 / (b_0 * (q2 + m2) * np.log((q2 + 4 * m2) / (Lambda ** 2)))

def T_1(k, q, phi, mg, a1, a2, m2_func):
    q2 = q
    qk_cos = np.sqrt(q) * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    G0 = G_p(q2, a1, a2)
    return alpha_D_plus * alpha_D_minus * G0 ** 2

def T_2(k, q, phi, mg, a1, a2, m2_func):
    q2 = q 
    qk_cos = np.sqrt(q) * k * np.cos(phi)
    qk_plus_squared = q2 / 4 + qk_cos + k ** 2
    qk_minus_squared = q2 / 4 - qk_cos + k ** 2
    alpha_D_plus = alpha_D(qk_plus_squared, mg, m2_func)
    alpha_D_minus = alpha_D(qk_minus_squared, mg, m2_func)
    factor = q2 + 9 * abs(k ** 2 - q2 / 4)
    G0 = G_p(q2, a1, a2)
    G_minus = G_p(factor, a1, a2)
    return alpha_D_plus * alpha_D_minus * G_minus * (2 * G0 - G_minus)

def integrand(y, x, mg, a1, a2, m2_func, q_val, sqrt_s):
    k = sqrt_s * x 
    phi = 2 * np.pi * y
    jacobian = 2 * np.pi * sqrt_s 
    return k * (T_1(k, q_val, phi, mg, a1, a2, m2_func) - T_2(k, q_val, phi, mg, a1, a2, m2_func)) * jacobian 

def sigma_tot(amp_value, s):
    return amp_value.imag / s * 0.389379323

def amp_calculation(diff_T, s, epsilon, t):
    alpha_pomeron = 1.0 + epsilon + alpha_prime * t
    regge_factor = (s**alpha_pomeron) * 1/(s0**(alpha_pomeron-1))
    return 1j * 8 * regge_factor * diff_T  

def differential_sigma(amp_value, s):
    amp_squared = amp_value.imag * amp_value.imag
    denominator = (16 * np.pi * s**2)
    return amp_squared / denominator * 0.389379323


In [5]:
import numpy as np
from scipy.integrate import quad
from numpy.polynomial.legendre import leggauss

def full_integral_quad(q2, mg, a1, a2, m2_func, sqrt_s, Nx=80):

    # Gauss-Legendre nodes for x ∈ [0,1]
    x_nodes, x_weights = leggauss(Nx)
    x_nodes = 0.5 * (x_nodes + 1.0)        # map from [-1,1] to [0,1]
    x_weights *= 0.5

    total_real = 0.0
    total_imag = 0.0
    total_err2 = 0.0

    for x, w in zip(x_nodes, x_weights):
        k = sqrt_s * x

        # integrand in φ
        def f(phi):
            val = T_1(k, q2, phi, mg, a1, a2, m2_func) \
                - T_2(k, q2, phi, mg, a1, a2, m2_func)
            return k * sqrt_s * val

        # real part
        r, er = quad(lambda ph: np.real(f(ph)), 0, 2*np.pi,
                     epsabs=1e-9, epsrel=1e-9)
        total_real += w * r
        total_err2 += (w * er)**2

        # imaginary part
        im, ei = quad(lambda ph: np.imag(f(ph)), 0, 2*np.pi,
                      epsabs=1e-9, epsrel=1e-9)
        total_imag += w * im
        total_err2 += (w * ei)**2

    T_value = total_real + 1j * total_imag
    T_error = np.sqrt(total_err2)

    return T_value, T_error

In [6]:
import numpy as np

def full_integral_quad_adaptive(q2, mg, a1, a2, m2_func, sqrt_s,
                                Nx_initial=1024,
                                max_Nx=4096,
                                tol=1e-3):
    """
    Adaptive Nx-doubling algorithm for T(q², s).
    
    Stops when:
        |T(Nx) - T(2Nx)| / |T(2Nx)| < tol
    """

    Nx = Nx_initial
    log = []

    # Compute first value
    T_prev, _ = full_integral_quad(
        q2=q2, mg=mg, a1=a1, a2=a2,
        m2_func=m2_func, sqrt_s=sqrt_s,
        Nx=Nx
    )
    log.append((Nx, T_prev))

    # Adaptive doubling loop
    while Nx < max_Nx:
        Nx_new = Nx * 2

        T_new, _ = full_integral_quad(
            q2=q2, mg=mg, a1=a1, a2=a2,
            m2_func=m2_func, sqrt_s=sqrt_s,
            Nx=Nx_new
        )
        log.append((Nx_new, T_new))

        # Compute convergence metric
        diff = T_new - T_prev
        rel_err = abs(diff) / max(abs(T_new), 1e-30)

        print(f"[adaptive Nx] Nx {Nx} → {Nx_new} : |ΔT|={abs(diff):.3e}, rel={rel_err:.3e}")

        if rel_err < tol:
            # Converged
            return {
                "T_value": T_new,
                "T_error": abs(diff),
                "Nx_used": Nx_new,
                "converged": True,
                "log": log
            }

        # Continue
        Nx = Nx_new
        T_prev = T_new

    # If reached max_Nx without convergence
    return {
        "T_value": T_prev,
        "T_error": np.nan,
        "Nx_used": Nx,
        "converged": False,
        "log": log
    }


In [7]:
def get_dif_sigma_adaptive(epsilon, mg, a1, a2, mg_model,
                           q2, sqrt_s=7000, tol=1e-3):

    s = sqrt_s**2

    # --- compute converged T(q²,s) ---
    result_T = full_integral_quad_adaptive(
        q2=q2,
        mg=mg, a1=a1, a2=a2,
        m2_func=mg_model,
        sqrt_s=sqrt_s,
        tol=tol,
        Nx_initial=1024,
        max_Nx=4096
    )

    T_val = result_T["T_value"]
    T_err = result_T["T_error"]
    converged = result_T["converged"]
    Nx_used = result_T["Nx_used"]

    # amplitude
    t = -q2
    amp_value = amp_calculation(T_val, s, epsilon, t)

    # dσ/dt
    dsigma = differential_sigma(amp_value, s)

    # propagate error: T → σ
    if abs(T_val) > 1e-30:
        rel_T_err = T_err / abs(T_val)
        dsigma_err = abs(dsigma) * 2 * rel_T_err
    else:
        dsigma_err = np.inf

    return {
        "q2": q2,
        "dsigma": dsigma,
        "dsigma_error": dsigma_err,
        "T_value": T_val,
        "T_error": T_err,
        "Nx_used": Nx_used,
        "converged": converged,
        "log": result_T["log"]
    }


In [8]:
dif_sigma_pl_atlas_7_q2 = []
dif_sigma_pl_atlas_7_values = []

for q2 in np.linspace(0.002, 0.2, 50):

    result = get_dif_sigma_adaptive(
        epsilon = ensemble_parameters["atlas"]["pl"]["epsilon"],
        mg      = ensemble_parameters["atlas"]["pl"]["mg"],
        a1      = ensemble_parameters["atlas"]["pl"]["a1"],
        a2      = ensemble_parameters["atlas"]["pl"]["a2"],
        mg_model = m2_pl,
        q2 = q2,
        tol = 1e-2
    )

    print(f"q2={q2:.5f}  dσ={result['dsigma']:.6f} ± {result['dsigma_error']:.2e} converged = {result["converged"]}")

    dif_sigma_pl_atlas_7_q2.append(q2)
    dif_sigma_pl_atlas_7_values.append(result["dsigma"])


[adaptive Nx] Nx 1024 → 2048 : |ΔT|=1.787e-02, rel=2.266e-03
q2=0.00200  dσ=435.944238 ± 1.98e+00 converged = True
[adaptive Nx] Nx 1024 → 2048 : |ΔT|=1.772e-02, rel=2.294e-03
q2=0.00604  dσ=403.708591 ± 1.85e+00 converged = True
[adaptive Nx] Nx 1024 → 2048 : |ΔT|=1.756e-02, rel=2.320e-03
q2=0.01008  dσ=373.770001 ± 1.73e+00 converged = True
[adaptive Nx] Nx 1024 → 2048 : |ΔT|=1.739e-02, rel=2.346e-03
q2=0.01412  dσ=345.973685 ± 1.62e+00 converged = True
[adaptive Nx] Nx 1024 → 2048 : |ΔT|=1.720e-02, rel=2.370e-03
q2=0.01816  dσ=320.174610 ± 1.52e+00 converged = True
[adaptive Nx] Nx 1024 → 2048 : |ΔT|=1.701e-02, rel=2.392e-03
q2=0.02220  dσ=296.237001 ± 1.42e+00 converged = True
[adaptive Nx] Nx 1024 → 2048 : |ΔT|=1.679e-02, rel=2.413e-03
q2=0.02624  dσ=274.033629 ± 1.32e+00 converged = True
[adaptive Nx] Nx 1024 → 2048 : |ΔT|=1.656e-02, rel=2.431e-03
q2=0.03029  dσ=253.445297 ± 1.23e+00 converged = True
[adaptive Nx] Nx 1024 → 2048 : |ΔT|=1.632e-02, rel=2.446e-03
q2=0.03433  dσ=234.

In [9]:
def add_differential_trace(fig, x, y, label, color='red', mg_model= 'log', legend=True, size = 4, width = 2):
    fig.add_trace(go.Scatter(
        x=x,
        y=y,
        mode='lines+markers',
        line=dict(color=color, width=width),
        marker=dict(size=size),
        name=f'{label}, {mg_model}',
        showlegend=legend,

    ))

def add_data_trace(fig, x, y, y_error, scale=1.0, color='black', size=4,
                           name=None, show_label=True, mode='markers'):
    fig.add_trace(go.Scatter(
        x=x,
        y=y * scale,
        mode=mode,
        marker=dict(color=color, size=size),
        error_y=dict(
            type='data',
            array=y_error * scale,
            visible=True
        ),
        name=name if show_label else None,
        showlegend=show_label
    ))

fig_atlas = go.Figure()


# for pl atlas
add_differential_trace(fig_atlas, dif_sigma_pl_atlas_7_q2, dif_sigma_pl_atlas_7_values,label='7 TeV', color='blue', mg_model='pl')

#data points
add_data_trace(fig_atlas, x_7_atlas, y_7_atlas, yerr_7_atlas, name='ATLAS 7 TeV', show_label=True, mode='markers')

# Atualiza layout
fig_atlas.update_layout(
    title='dσ/dt vs. |t| - Log and PL models in ATLAS',
    xaxis_title='|t| (GeV²)',
    yaxis_title='dσ/dt (mb/GeV²)',
    yaxis_type='log',
    legend_title='Mass Model',
    plot_bgcolor='white',
    hovermode='x unified'
)

fig_atlas.update_xaxes(gridcolor='lightgray')
fig_atlas.update_yaxes(gridcolor='lightgray')

# fig_atlas.show(renderer='browser')


In [10]:
# # =============================================================
# #  PLOT BORN SIGMA TOT BORN
# # =============================================================


data_sigma_tot_atlas = pd.read_csv(
    "../../../../../data/sigma_tot_2/ensemble_StRh_atlas.dat",
    delim_whitespace=True,
    header=None,
    nrows=70
)

x_sigma_tot_atlas = data_sigma_tot_atlas[0].to_numpy()
y_sigma_tot_atlas = data_sigma_tot_atlas[1].to_numpy()
y_error_sigma_tot_atlas = data_sigma_tot_atlas[2].to_numpy()

lst_born_amp = []

start_sqrt_s = 6000
max_sqrt_s = 13010
step = 800

def add_total_trace(fig, x, y, color='red', label='', line_style='solid', legend=True, size = 3, width = 2):
    fig.add_trace(go.Scatter(
        x=x,
        y=y,
        mode='lines+markers',
        line=dict(color=color, width=width, dash=line_style),
        marker=dict(size=size),
        name = label, 
        showlegend=legend
    ))


/tmp/ipykernel_10294/3011614697.py:6: FutureWarning:

The 'delim_whitespace' keyword in pd.read_csv is deprecated and will be removed in a future version. Use ``sep='\s+'`` instead



In [18]:
def get_sigma_tot_quad(
        epsilon, mg, a1, a2, mg_model,
        start_sqrt_s=6000,
        max_sqrt_s=13000,
        step=1000,
        tol=1e-2,
        Nx_initial=128,
        max_Nx=4096):
    """
    Compute σ_tot(s) using the quad-based 2D integral with
    adaptive Nx doubling until convergence.

    σ_tot = Im[A(s, t=0)] / s
    """

    lst_sqrt_s = []
    lst_sigma_tot = []
    lst_sigma_tot_error = []
    lst_relative_error = []
    lst_integral_values = []

    q2 = 0.0        # forward scattering amplitude
    t  = 0.0        # t = 0 for optical theorem

    sqrt_s = start_sqrt_s

    while sqrt_s <= max_sqrt_s + 0.5 * step:

        s = sqrt_s**2

        # ------------------------------------------------------------------
        # 1) Adaptive Nx-doubling integral for T(q²=0, s)
        # ------------------------------------------------------------------
        result_T = full_integral_quad_adaptive(
            q2=q2,
            mg=mg, a1=a1, a2=a2,
            m2_func=mg_model,
            sqrt_s=sqrt_s,
            tol=tol,
            Nx_initial=Nx_initial,
            max_Nx=max_Nx
        )

        T_value = result_T["T_value"]
        T_error = result_T["T_error"]
        Nx_used = result_T["Nx_used"]
        converged = result_T["converged"]

        lst_integral_values.append(T_value)

        # ------------------------------------------------------------------
        # 2) Optical theorem: σ_tot = Im A(s, t=0) / s
        # ------------------------------------------------------------------
        born_amp = amp_calculation(T_value, s, epsilon, t)
        sigma_value = sigma_tot(born_amp, s)

        # ------------------------------------------------------------------
        # 3) Error propagation
        # ------------------------------------------------------------------
        if abs(T_value) > 1e-30:
            rel_T_err = T_error / abs(T_value)
            sigma_err = abs(sigma_value) * rel_T_err
        else:
            rel_T_err = np.inf
            sigma_err = np.inf

        lst_sqrt_s.append(sqrt_s)
        lst_sigma_tot.append(sigma_value)
        lst_sigma_tot_error.append(sigma_err)
        lst_relative_error.append(rel_T_err)

        # ------------------------------------------------------------------
        # 4) Console output for monitoring
        # ------------------------------------------------------------------
        print("─" * 85)
        print(f"√s = {sqrt_s:.3f} GeV")
        print(f"T integral (q²=0):      {T_value:+.10e}")
        print(f"T error (adaptive Nx):  {T_error:.3e}")
        print(f"Converged:              {converged}   Nx={Nx_used}")
        print(f"σ_tot(s):               {sigma_value:.6e} ± {sigma_err:.3e}")
        print("─" * 85)

        sqrt_s += step

    # ----------------------------------------------------------------------
    # Return all results as arrays
    # ----------------------------------------------------------------------
    return {
        "sqrt_s": np.array(lst_sqrt_s),
        "sigma_tot": np.array(lst_sigma_tot),
        "sigma_tot_error": np.array(lst_sigma_tot_error),
        "relative_error": np.array(lst_relative_error),
        "integral_values": np.array(lst_integral_values),
    }


In [19]:
results_sigma_tot_pl_atlas = get_sigma_tot_quad(
    epsilon = ensemble_parameters["atlas"]["pl"]["epsilon"],
    mg      = ensemble_parameters["atlas"]["pl"]["mg"],
    a1      = ensemble_parameters["atlas"]["pl"]["a1"],
    a2      = ensemble_parameters["atlas"]["pl"]["a2"],
    mg_model = m2_pl,
    Nx_initial=1024    # recommended for σ_tot
)

sigma_tot_pl_atlas_values = results_sigma_tot_pl_atlas["sigma_tot"]
lst_sqrt_s                 = results_sigma_tot_pl_atlas["sqrt_s"]
sigma_tot_errors           = results_sigma_tot_pl_atlas["sigma_tot_error"]


[adaptive Nx] Nx 1024 → 2048 : |ΔT|=1.067e-02, rel=1.339e-03
─────────────────────────────────────────────────────────────────────────────────────
√s = 6000.000 GeV
T integral (q²=0):      +7.9659873260e+00+0.0000000000e+00j
T error (adaptive Nx):  1.067e-02
Converged:              True   Nx=2048
σ_tot(s):               9.197972e+01 ± 1.232e-01
─────────────────────────────────────────────────────────────────────────────────────
[adaptive Nx] Nx 1024 → 2048 : |ΔT|=1.794e-02, rel=2.252e-03
─────────────────────────────────────────────────────────────────────────────────────
√s = 7000.000 GeV
T integral (q²=0):      +7.9659886520e+00+0.0000000000e+00j
T error (adaptive Nx):  1.794e-02
Converged:              True   Nx=2048
σ_tot(s):               9.414003e+01 ± 2.120e-01
─────────────────────────────────────────────────────────────────────────────────────
[adaptive Nx] Nx 1024 → 2048 : |ΔT|=1.131e-02, rel=1.420e-03
─────────────────────────────────────────────────────────────────────────

In [20]:


fig = go.Figure()

add_total_trace(fig, lst_sqrt_s, sigma_tot_pl_atlas_values, color='blue', label='PL Atlas', line_style='solid')

#-----------------------------------------------------------------------------------------------
#----

add_data_trace(fig, x_sigma_tot_atlas, y_sigma_tot_atlas, y_error_sigma_tot_atlas, name='ATLAS', show_label=True, mode='markers')

fig.update_layout(
    title = 'σ_tot vs. √s - Ensemble Atlas and Totem in Log and PL model',
    xaxis=dict(
        title='√s [GeV]',
        type='log',
        range=[np.log10(2000), np.log10(14000)],
    ),
    yaxis=dict(
        title='σ_tot [mb]',
        range=[80, 125]
    ),
    showlegend=True,
    legend=dict(
        title='Ensembles'
    ),
    plot_bgcolor='white',
    hovermode='x unified'
)
    
fig.update_xaxes(gridcolor='lightgray')
fig.update_yaxes(gridcolor='lightgray')

# fig.show(renderer="browser")